In [ ]:
# N is the length of Text pattern
# M is the length of pattern
import math
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector


text: str = "1011"
pattern: str = "01"

def prepare_input(text: str, pattern: str) -> tuple[QuantumCircuit, QuantumRegister, QuantumRegister, QuantumRegister]:
        # T = t0t1t2t3 = 1011
    # t_i <=> q_i,so t0 = q0
    # Paper/logical order: \(t_0,t_1,t_2,t_3\)
    # Qiskit printed ket order: \(q_3,q_2,q_1,q_0\)
    N: int = len(text)
    M: int = len(pattern)
    index_size: int = math.ceil(math.log2(N))

    print("N: ", N)
    print("M: ", M)
    print("i: ", index_size)


    # Index register has n = log2(N) qubits.

    # Text register: N qubits
    # Pattern register: M qubits
    # Index register: log2(N) qubits
    # Text and pattern are prepared as basis states.
    # Only the index register is put into uniform superposition.

    # The goal is to shift the text register using the index register we will create soon.


    index_reg = QuantumRegister(index_size, "index") # |k>
    text_reg = QuantumRegister(N, "text")            # |T>
    pattern_reg = QuantumRegister(M, "pattern")      # |P>

    qc = QuantumCircuit(index_reg, text_reg, pattern_reg)

    # Paper convention: text_reg[i] stores t_i.
    # Qiskit displays basis-state labels as |q_(N-1)...q_0|,
    # so printed bitstrings appear reversed relative to T = t0 t1 ... t_(N-1).
    def transform_text_reg(text):
        for i, bit in enumerate(text):
            if bit == "1":
                qc.x(text_reg[i])
            print("i: ", i)
            print("bit: ", bit)

    def transform_pattern_reg(pattern):
        for i, bit in enumerate(pattern):
            if bit == "1":
                qc.x(pattern_reg[i])
            print("i: ", i)
            print("bit: ", bit)

    print("------------------")
    transform_text_reg(text)
    print("------------------")

    transform_pattern_reg(pattern)

    # Put index register into uniform superposition
    qc.h(index_reg)

    return qc, index_reg, text_reg, pattern_reg

def verify_hadamard_uniformity(qc: QuantumCircuit, index_reg):
    state = Statevector.from_instruction(qc)

    index_qubits = [qc.find_bit(q).index for q in index_reg]

    print(state.probabilities_dict(qargs=index_qubits))

qc, index_reg, text_reg, pattern_reg = prepare_input(text, pattern)

print(qc.draw('text', fold=-1))


print(len(index_reg))

# Left shifts the qubits of the text register by k positions.
def cyclic_shift_operator(qc, index_reg, text_reg):
    """Applies controlled cyclic left shift on a 4-qubit text register.

    - S1: shift by 1, controlled by index_reg[0] (k0, weight 2^0 = 1)
    - S2: shift by 2, controlled by index_reg[1] (k1, weight 2^1 = 2)
    """
    # ---S1 (shift by 1 if k0 == 1)---
    qc.cswap(index_reg[0], text_reg[0], text_reg[1])
    qc.cswap(index_reg[0], text_reg[1], text_reg[2])
    qc.cswap(index_reg[0], text_reg[2], text_reg[3])

    # ---S2 (shift by 2 if k1 == 1)---
    qc.cswap(index_reg[1], text_reg[0], text_reg[2])
    qc.cswap(index_reg[1], text_reg[1], text_reg[3])

# But we only want it to happen if k0 = 1
# Note for index_size = 2 such that k = 2k1 + k0 
# This means that if we have 1/2( |00> + |01> + |10> + |11> ) then |01> and |11> has k0 = 1
# So we replace with conditional 
print("---------------------------------------")
cyclic_shift_operator(qc, index_reg, text_reg)
#print(qc.draw('text', fold=-1))
qc.draw("mpl")

# Implement XOR
# After cycle shift all candidate bits are on the first M qubits of the text register.
# We apply the CNOT with each M first qubits between text_reg and pattern_reg
# Then we check if the outcome is 00 for any CNOT  

def apply_xor(qc, text_reg, pattern_reg):
    """Applies the XOR function via controlled NOTs on the pattern register with the text register.
    """
    for i in range(len(pattern_reg)):
        qc.cx(text_reg[i], pattern_reg[i])

apply_xor(qc, text_reg, pattern_reg)
qc.draw("mpl")
#print(qc.draw('text', fold=-1))

def debug_xor(qc):
    # Ordered oppositely |pattern> |text> |shift>
    sv = Statevector.from_instruction(qc)
    for bitstring, amp in sv.to_dict().items():
        if abs(amp) > 1e-5:
            print("bitstring: ", bitstring)
            print("amp: ", amp)
            pattern_part = bitstring[:2][::-1] # Reverse back so they appear in correct order
            text_part = bitstring[2:6][::-1] # Revser back so they appear in correct order
            shift_part = bitstring[6:8]
            k = int(shift_part, 2)
            print(f"Shift k={k} ({shift_part}): pattern_reg = '{pattern_part}'")





N:  4
M:  2
i:  2
------------------
i:  0
bit:  1
i:  1
bit:  0
i:  2
bit:  1
i:  3
bit:  1
------------------
i:  0
bit:  0
i:  1
bit:  1
           ┌───┐
  index_0: ┤ H ├
           ├───┤
  index_1: ┤ H ├
           ├───┤
   text_0: ┤ X ├
           └───┘
   text_1: ─────
           ┌───┐
   text_2: ┤ X ├
           ├───┤
   text_3: ┤ X ├
           └───┘
pattern_0: ─────
           ┌───┐
pattern_1: ┤ X ├
           └───┘
2
---------------------------------------
bitstring:  00111001
amp:  (0.4999999999999999+0j)
Shift k=1 (01): pattern_reg = '00'
bitstring:  01011110
amp:  (0.4999999999999999+0j)
Shift k=2 (10): pattern_reg = '10'
bitstring:  01101111
amp:  (0.4999999999999999+0j)
Shift k=3 (11): pattern_reg = '10'
bitstring:  11110100
amp:  (0.4999999999999999+0j)
Shift k=0 (00): pattern_reg = '11'
